# 03 - Correlación de Spearman (interactivo, telemetría fija y CropX)

Widget para elegir finca y tipos de variables; una celda por etapa y por base.
Si una finca no tiene datos en la etapa, se muestra una figura con el aviso "Sin datos".

> Ejecutar las celdas en orden. Usa `config.py` y `funciones_comunes.py` (misma carpeta).

In [ ]:
# =============================================================================
# LIBRERÍAS
# =============================================================================
import re
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec
import seaborn as sns
import ipywidgets as widgets
from scipy import stats
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
# Gradientes de color sobre filas sin datos válidos (sensor descartado): aviso inofensivo de pandas
warnings.filterwarnings("ignore", message="All-NaN slice encountered")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

In [ ]:
# =============================================================================
# CONFIGURACION Y FUNCIONES COMUNES (editar config.py y reiniciar el kernel)
# =============================================================================
from config import *
from funciones_comunes import (_limites_rango, obtener_rangos, periodo_finca, filtrar_etapa, columnas_analisis,
                               cargar_finca, cargar_bd_fija, cargar_cropx, cargar_bd_cropx,
                               mostrar_sin_datos, contar_por_banda, _ajustar_slider)

In [ ]:
# Telemetría fija y CropX de todas las fincas, recortadas al periodo de sus etapas (incluye Descanso)
BD_FIJA_FILTERED = cargar_bd_fija()
BD_CROPX = cargar_bd_cropx()
BASES = {NOMBRE_BD_FIJA: BD_FIJA_FILTERED, NOMBRE_BD_CROPX: BD_CROPX}


Cargando datos de la finca: La_Orquidea


Se eliminaron 0 filas con valores NaN en la columna 'timestamp' y 0 filas con índice NaN.
Datos cargados para La_Orquidea: 17447 filas y 43 columnas.

Cargando datos de la finca: Venecia
Se eliminaron 0 filas con valores NaN en la columna 'timestamp' y 0 filas con índice NaN.
Datos cargados para Venecia: 17199 filas y 43 columnas.

Cargando datos de la finca: Puerto_Arturo
Se eliminaron 0 filas con valores NaN en la columna 'timestamp' y 0 filas con índice NaN.
Datos cargados para Puerto_Arturo: 17064 filas y 43 columnas.
La_Orquidea: fechas en reloj de 12 h sin AM/PM -> 8733 registros pasados a la tarde según el orden del archivo.
La_Orquidea: 17449 registros CropX, 2025-10-21 10:00:00 a 2026-10-19 22:30:00
Venecia: fechas en reloj de 12 h sin AM/PM -> 8736 registros pasados a la tarde según el orden del archivo.
Venecia: 17450 registros CropX, 2025-10-21 11:00:00 a 2026-10-19 23:30:00
Puerto_Arturo: fechas en reloj de 12 h sin AM/PM -> 3360 registros pasados a la tarde según el orden

## Correlación

In [ ]:
# =============================================================================
# CORRELACION DE SPEARMAN POR FINCA, TIPO DE VARIABLE Y ETAPA
# =============================================================================
OPCION_TODAS = "TODAS"


def seleccionar_variables(df: pd.DataFrame, tipos: tuple[str, ...], grupos: dict[str, list[str]]) -> list[str]:
    """Lista de columnas a correlacionar según los tipos elegidos.

    Args:
        df (pd.DataFrame): Telemetría de una finca (índice temporal).
        tipos (tuple[str, ...]): Claves de `grupos` o ("TODAS",).
        grupos (dict[str, list[str]]): {tipo: columnas}, p. ej. TIPO_VARS_BD_FIJA o GRUPOS_CROPX.

    Returns:
        list[str]: Columnas existentes en `df`, sin duplicados y en el orden de los tipos.
    """
    if OPCION_TODAS in tipos:
        tipos = tuple(grupos)
    columnas = [c for t in tipos for c in grupos[t]]
    return [c for c in dict.fromkeys(columnas) if c in df.columns]


def correlacion_spearman(datos: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    """Matriz de Spearman descartando columnas vacías o constantes.

    Args:
        datos (pd.DataFrame): Columnas numéricas a correlacionar.

    Returns:
        tuple[pd.DataFrame, list[str]]: Matriz de correlación y columnas descartadas.
    """
    validas = datos.dropna(axis=1, how="all")
    validas = validas.loc[:, validas.nunique() > 1]       # constantes -> rho indefinido
    descartadas = datos.columns.difference(validas.columns, sort=False).tolist()
    return validas.corr(method="spearman"), descartadas


def pares_correlacionados(corr: pd.DataFrame, umbral: float = 0.6) -> pd.DataFrame:
    """Pares de variables con |rho| >= umbral (triángulo superior, sin la diagonal).

    Args:
        corr (pd.DataFrame): Matriz de correlación.
        umbral (float): Valor absoluto mínimo de rho.

    Returns:
        pd.DataFrame: Columnas [Variable 1, Variable 2, rho], ordenado por |rho| descendente.
    """
    superior = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    pares = superior.stack().rename("rho").reset_index()
    pares.columns = ["Variable 1", "Variable 2", "rho"]
    pares = pares[pares["rho"].abs() >= umbral]
    return pares.reindex(pares["rho"].abs().sort_values(ascending=False).index).reset_index(drop=True)


def graficar_heatmap(corr: pd.DataFrame, titulo: str) -> None:
    """Heatmap del triángulo inferior; anota valores solo si hay 20 variables o menos.

    Args:
        corr (pd.DataFrame): Matriz de correlación.
        titulo (str): Título de la figura.

    Returns:
        None: Muestra la figura.
    """
    n = len(corr)
    fig, ax = plt.subplots(figsize=(max(8, 0.55 * n), max(6, 0.45 * n)))
    sns.heatmap(corr, mask=np.triu(np.ones_like(corr, dtype=bool), k=1),
                cmap="coolwarm", vmin=-1, vmax=1, center=0, square=True,
                annot=n <= 20, fmt=".2f", annot_kws={"size": 8},
                linewidths=0.5, cbar_kws={"label": "rho de Spearman", "shrink": 0.7}, ax=ax)
    ax.set_title(titulo, fontsize=13)
    plt.tight_layout()
    plt.show()


def panel_correlacion(bd: dict[str, pd.DataFrame], grupos: dict[str, list[str]], nombre_bd: str,
                      etapa: str | None = None, umbral: float = 0.6) -> None:
    """Widget: elige finca y tipos de variables, y muestra la correlación de la etapa.

    Args:
        bd (dict[str, pd.DataFrame]): {finca: datos con índice temporal} (BD_FIJA_FILTERED o BD_CROPX).
        grupos (dict[str, list[str]]): Tipos de variables de la base (TIPO_VARS_BD_FIJA o GRUPOS_CROPX).
        nombre_bd (str): Nombre de la base para los títulos.
        etapa (str | None): Nombre de la etapa o None para el ciclo completo.
        umbral (float): |rho| mínimo para listar pares correlacionados.

    Returns:
        None: Muestra los widgets, el heatmap y la tabla de pares.
    """
    if not bd:
        mostrar_sin_datos(f"Spearman - {nombre_bd}", "No hay fincas cargadas en esta base")
        return
    w_finca = widgets.Dropdown(options=list(bd), description="Finca")
    w_tipos = widgets.SelectMultiple(options=[OPCION_TODAS, *grupos],
                                     value=(OPCION_TODAS,), rows=8, description="Tipos")

    def _actualizar(finca, tipos):
        if not tipos:
            print("Selecciona al menos un tipo.")
            return
        nombre_etapa = etapa or "Ciclo completo"
        titulo = f"Spearman - {nombre_bd} - {finca} - {nombre_etapa}"
        if etapa is not None and etapa not in obtener_rangos(FINCAS_POR_NOMBRE[finca]):
            mostrar_sin_datos(titulo, f"{finca} no tiene la etapa '{etapa}'")
            return
        df = filtrar_etapa(bd[finca], finca, etapa)
        if df.empty:
            mostrar_sin_datos(titulo)
            return

        corr, descartadas = correlacion_spearman(df[seleccionar_variables(df, tipos, grupos)])
        if descartadas:
            print("Descartadas (vacías o constantes):", ", ".join(descartadas))
        if corr.shape[0] < 2:
            mostrar_sin_datos(titulo, "Menos de 2 variables válidas: no hay correlación que calcular")
            return

        graficar_heatmap(corr, f"{titulo} - n = {len(df)}")
        pares = pares_correlacionados(corr, umbral)
        display(Markdown(f"**Pares con |rho| >= {umbral}: {len(pares)}**"))
        if not pares.empty:
            display(pares.style.background_gradient(subset=["rho"], cmap="coolwarm", vmin=-1, vmax=1)
                         .format({"rho": "{:.2f}"}))

    salida = widgets.interactive_output(_actualizar, {"finca": w_finca, "tipos": w_tipos})
    display(widgets.HBox([w_finca, w_tipos]), salida)

## Correlación telemetría fija

In [ ]:
panel_correlacion(BD_FIJA_FILTERED, TIPO_VARS_BD_FIJA, NOMBRE_BD_FIJA)   # Ciclo completo

Output()

In [ ]:
panel_correlacion(BD_FIJA_FILTERED, TIPO_VARS_BD_FIJA, NOMBRE_BD_FIJA, "Descanso")

Output()

In [ ]:
panel_correlacion(BD_FIJA_FILTERED, TIPO_VARS_BD_FIJA, NOMBRE_BD_FIJA, "Pre-siembra")

Output()

In [ ]:
panel_correlacion(BD_FIJA_FILTERED, TIPO_VARS_BD_FIJA, NOMBRE_BD_FIJA, "Vegetativa")

Output()

In [ ]:
panel_correlacion(BD_FIJA_FILTERED, TIPO_VARS_BD_FIJA, NOMBRE_BD_FIJA, "Floración")

Output()

In [ ]:
panel_correlacion(BD_FIJA_FILTERED, TIPO_VARS_BD_FIJA, NOMBRE_BD_FIJA, "Cosecha")

Output()

In [ ]:
panel_correlacion(BD_FIJA_FILTERED, TIPO_VARS_BD_FIJA, NOMBRE_BD_FIJA, "Descanso2")

Output()

## Correlación CropX

In [ ]:
panel_correlacion(BD_CROPX, GRUPOS_CROPX, NOMBRE_BD_CROPX)   # Ciclo completo

Output()

In [ ]:
panel_correlacion(BD_CROPX, GRUPOS_CROPX, NOMBRE_BD_CROPX, "Descanso")

Output()

In [ ]:
panel_correlacion(BD_CROPX, GRUPOS_CROPX, NOMBRE_BD_CROPX, "Pre-siembra")

Output()

In [ ]:
panel_correlacion(BD_CROPX, GRUPOS_CROPX, NOMBRE_BD_CROPX, "Vegetativa")

Output()

In [ ]:
panel_correlacion(BD_CROPX, GRUPOS_CROPX, NOMBRE_BD_CROPX, "Floración")

Output()

In [ ]:
panel_correlacion(BD_CROPX, GRUPOS_CROPX, NOMBRE_BD_CROPX, "Cosecha")

Output()

In [ ]:
panel_correlacion(BD_CROPX, GRUPOS_CROPX, NOMBRE_BD_CROPX, "Descanso2")

Output()